# Listener Prior (Dual Dataset) - Colab GPU Training (Public Repo)

This notebook:
- clones the repo into `/content/listener-prior`
- installs dependencies (pins `datasets<4.0.0` so MultiWOZ/DailyDialog script datasets load)
- loads Hugging Face token from Colab Secrets (`HF_TOKEN`)
- trains on **MultiWOZ 2.2 + DailyDialog**
- evaluates on the **test** split each epoch and saves `encoder_best/` when improved
- writes outputs to Google Drive so they persist


In [ ]:
# --- CONFIG: set your GitHub repo here ---
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>.git"  # <- edit
PROJECT_DIR = "/content/listener-prior"

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!ls -la

In [ ]:
# Install deps. We intentionally do NOT pin torch here; Colab already has a CUDA build.
import pathlib

req = pathlib.Path("requirements.txt").read_text().splitlines()
req_no_torch = [r for r in req if r.strip() and not r.strip().startswith("torch")]
pathlib.Path("/tmp/requirements_no_torch.txt").write_text("\n".join(req_no_torch) + "\n")

!python -m pip install -U pip
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade --force-reinstall

import datasets
print("datasets:", datasets.__version__)
assert tuple(int(x) for x in datasets.__version__.split(".")[:1]) < (4,), "datasets must be < 4.0.0; restart runtime after install"

In [ ]:
# If the assert above failed, go to Runtime -> Restart runtime, then rerun from the top.
pass

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hugging Face token (recommended).
# In Colab: Tools -> Secrets -> add HF_TOKEN.
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

# Clear any stale/expired tokens that might already exist in the environment.
for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')

In [ ]:
import datetime
run_id = datetime.datetime.now().strftime('dual_run_%Y%m%d_%H%M%S')
OUTPUT_DIR = f"/content/drive/MyDrive/listener_prior_runs/{run_id}"
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Train. This evaluates on TEST each epoch and updates encoder_best/ when improved.
!python scripts/train_dual_epoch_test.py \
  --output_dir "$OUTPUT_DIR" \
  --device auto \
  --epochs 6 \
  --batch_size 32 \
  --learning_rate 4.3e-5 \
  --weight_decay 0.01 \
  --adam_beta1 0.95 \
  --adam_beta2 0.98 \
  --adam_eps 1e-8 \
  --grad_accum_steps 2 \
  --warmup_ratio 0.0 \
  --history_turns 6 \
  --val_ratio 0.05 \
  --test_ratio 0.15 \
  --max_dialogs_multiwoz 0 \
  --max_dialogs_dailydialog 0

In [ ]:
# Quick offline demo using the best encoder.
RUN_DIR = OUTPUT_DIR
!python -m src.demo_offline --run "$RUN_DIR" --encoder_subdir encoder_best